# End-to-End Face Detection & Recognition Pipeline

This notebook walks through the full pipeline built on top of [biubug6/Pytorch_Retinaface](https://github.com/biubug6/Pytorch_Retinaface):

1. **Detection** — RetinaFace locates faces + 5-point landmarks in an image.
2. **Alignment** — faces are cropped and warped to a canonical pose using the landmarks.
3. **Embedding** — aligned faces are converted to embedding vectors. Three embedding models are compared: FaceNet (VGGFace2), ArcFace (InsightFace `buffalo_l`), and AdaFace (`ir_50`, WebFace4M).
4. **Indexing & search** — embeddings are stored in a FAISS index (wrapped with LangChain for metadata-aware similarity search) to support face deduplication and identification.

All commands below assume the working directory is the repo root and that `weights/` and `pretrained/` contain the required checkpoints (see README for download links).

## 1. Dataset sanity check

Load a WIDER-FACE-format dataset and visualize one annotated sample (bounding box + 5 landmarks).

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

from data.wider_face import WiderFaceDetection

dataset = WiderFaceDetection("data/my_faces/label.txt")
print(f"Total images: {len(dataset)}")

img, ann = dataset[0]
print("Image tensor shape:", img.shape)
print("Annotations shape:", ann.shape)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import torch

img, ann = dataset[9]
if isinstance(img, torch.Tensor):
    img = img.numpy()
img_rgb = cv2.cvtColor(img.astype("uint8"), cv2.COLOR_BGR2RGB)

for face in ann:
    x1, y1, x2, y2 = map(int, face[:4])
    cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
    for i in range(5):
        lx, ly = int(face[4 + 2 * i]), int(face[5 + 2 * i])
        cv2.circle(img_rgb, (lx, ly), 3, (255, 0, 0), -1)

plt.imshow(img_rgb)
plt.axis("off")
plt.show()

## 2. Training / fine-tuning

RetinaFace can be fine-tuned on a custom labeled set (here `data/my_faces/label.txt`, generated from InsightFace auto-annotation). Example commands (not executed by default — training takes hours):

```bash
python train.py --network mobile0.25 --training_dataset ./data/my_faces/label.txt
python train.py --network resnet50   --training_dataset ./data/my_faces/label.txt
```

## 3. Detection

Run the trained/pretrained RetinaFace model on a sample image.

In [ ]:
!python detect.py --network resnet50

## 4. Embedding model comparison

Three embedding backbones were evaluated on the same aligned-face crops to pick the best one for the recognition pipeline.

### 4a. FaceNet (VGGFace2)

In [ ]:
!python embeddings.py

### 4b. ArcFace (InsightFace `buffalo_l`)

In [ ]:
import insightface

arcface_model = insightface.model_zoo.get_model("buffalo_l")
arcface_model.prepare(ctx_id=0)  # GPU: 0, CPU: -1
print("ArcFace model loaded")

### 4c. AdaFace (`ir_50`, WebFace4M) — chosen embedding model

AdaFace gave the most consistent cosine-similarity separation between same/different identities on our validation crops, so it's the embedder used by the production pipeline (`face_pipeline_adaface.py`).

In [ ]:
!python adaface_embeddings.py

## 5. Build the FAISS similarity index

Embeddings are written to a FAISS index and wrapped in a LangChain vector store so each vector carries a `face_id` for metadata-aware lookups.

In [ ]:
!python database.py
!python langchain_faiss.py

## 6. Full pipeline: detect → align → embed → dedup-check → store

`face_pipeline_adaface.py` runs the complete flow on a new image: detect faces, align them, embed with AdaFace, search the FAISS index for a near-duplicate (cosine similarity threshold), and either flag the match or register a new `face_id`.

In [ ]:
!python face_pipeline_adaface.py

## 7. Evaluation

Quantitative precision/recall/F1 evaluation of the recognition pipeline on a held-out gallery/probe split — see `evaluation.py` and `evaluation_report.txt` for full results.

In [ ]:
!python evaluation.py